<a href="https://colab.research.google.com/github/wilsonmarundaa-netizen/updated-flyRank-intern/blob/main/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wilsonmarundaa-netizen/updated-flyRank-intern/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the

claim? Constructive tone.*

Where does the trend_direction label come from?

During my feature and target audit, I found that the trend_direction label is derived from the relationship between impressions_last_30d and impressions_90d. These same variables are also available as features in my model.

This raises a potential target-leakage concern: the model may be learning information that directly determines the target rather than independently learning signals associated with trend direction.

I would therefore want to clarify how trend_direction was constructed, including the exact calculation used to generate the label and the time period from which the underlying data was collected.



In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
!wget https://raw.githubusercontent.com/wilsonmarundaa-netizen/updated-flyRank-intern/main/data/raw/content_refresh_anonymized.csv

import pandas as pd

df = pd.read_csv('content_refresh_anonymized.csv')

df["down_trend"] = (df["trend_direction"] == "down").astype(int)

df["down_trend"].head()

df = df.dropna()

df.columns

--2026-08-26 03:34:44--  https://raw.githubusercontent.com/wilsonmarundaa-netizen/updated-flyRank-intern/main/data/raw/content_refresh_anonymized.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6727670 (6.4M) [text/plain]
Saving to: ‘content_refresh_anonymized.csv’

content_refresh_ano 100%[===================>]   6.42M  --.-KB/s    in 0.08s   

2026-08-26 03:34:44 (84.4 MB/s) - ‘content_refresh_anonymized.csv’ saved [6727670/6727670]



Index(['content_id', 'client_id', 'search_volume', 'competition',
       'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count',
       'char_count', 'provider_used', 'model_used', 'impressions_90d',
       'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
       'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
       'days_with_impressions', 'days_with_sessions', 'impressions_last_30d',
       'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d',
       'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier',
       'age_tier_order', 'days_since_last_update', 'freshness_tier',
       'word_count_tier', 'char_count_tier', 'ctr', 'avg_position',
       'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier',
       'position_tier', 'trend_direction', 'trend_pct', 'down_trend'],
      dtype='object')

The research paper mentioned the company uses scroll depth as part of the health score. My data has scroll_rate. Is it the same as the scroll_depth. What does scroll_depth represent and where does it come from?

If I use the scroll rate on my model and baseline would it change the results?

In [3]:
#investigate scroll_rate from random value
df['scroll_rate'].iloc[98]

np.float64(6.67)

In [4]:
df["pageviews_90d"].corr(df["scroll_rate"])

np.float64(-0.11833625226087728)

INITIAL MODEL

In [5]:
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.metrics import f1_score,precision_score,recall_score

features =['impressions_90d',
       'clicks_90d', 'pageviews_90d','cpc','sessions_90d', 'users_90d','impressions_last_30d', 'engagement_rate','search_volume']

X = df[features]
y = df["down_trend"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

test_results = X_test.copy()
test_results["model_predictions"] = model.predict(X_test)
test_results["actual_values"] = y_test


test_results["baseline_prediction"] = np.where(
    test_results["impressions_last_30d"] / 30 > test_results["impressions_90d"] / 90,
    0,  #up
    1  #down
)



f1 = f1_score(y_test,test_results["model_predictions"])
precision = precision_score(y_test,test_results["model_predictions"])
recall = recall_score(y_test,test_results["model_predictions"])


print(f"f1: {f1}")
print(f"precision: {precision}")
print(f"recall: {recall}")


f1: 0.8559790514983998
precision: 0.7598140495867769
recall: 0.9800133244503664


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


REVIEWED MODEL

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score,precision_score,recall_score
from sklearn.linear_model import LogisticRegression


features =[
       'scroll_rate','clicks_90d', 'pageviews_90d','cpc','sessions_90d', 'users_90d', 'engagement_rate','search_volume',]

new_X = df[features]
new_y = df["down_trend"]

new_X_train, new_X_test, new_y_train, new_y_test = train_test_split(new_X, new_y, test_size=0.3, random_state=42, stratify=y)



In [7]:
new_model = LogisticRegression(max_iter=1000)

new_model.fit(new_X_train, new_y_train)



LogisticRegression(max_iter=1000)

In [8]:
new_test_results = new_X_test.copy()
new_test_results["model_predictions"] = new_model.predict(new_X_test)
new_test_results["actual_values"] = new_y_test



new_f1 = f1_score(new_y_test,new_test_results["model_predictions"])
new_precision = precision_score(new_y_test,new_test_results["model_predictions"])
new_recall = recall_score(new_y_test,new_test_results["model_predictions"])


print(f"new_f1: {new_f1},intial: {f1}")
print(f"new_precision: {new_precision},initial: {precision}")
print(f"new_recall: {new_recall}, initial: {recall}")

new_f1: 0.8147540983606557,intial: 0.8559790514983998
new_precision: 0.6905974988420565,initial: 0.7598140495867769
new_recall: 0.9933377748167888, initial: 0.9800133244503664


In [12]:

import os

os.makedirs("saved_data", exist_ok=True)


df.to_pickle("saved_data/data.pkl")

In [9]:

import joblib

joblib.dump(new_model, "my_model.pkl")

['my_model.pkl']

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

I checked my final features to make sure they do not contain information used to create the target or information from the future. I found that impressions_last_30d and impressions_90d were used to create trend_direction, so including them would cause target leakage because the model could learn the rule used to generate the label. I therefore removed these features and used the remaining features to test whether they provide independent signals for predicting trend direction.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

After removing the leaked features, the model's performance stayed almost the same. This suggests that the other features are still useful for predicting trend direction, but more testing is needed to know how well the model will perform on new data because it doesnt prove that the model will perform well in future data.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.